# Connect Four Game-Playing Agent Demo

This notebook demonstrates our Connect Four AI agent built using adversarial search techniques:
- **Minimax** (baseline, no pruning)
- **Alpha-Beta Pruning** with center-first move ordering
- **Iterative Deepening** with time budget
- **Transposition Table** for caching evaluated positions

We show how to create and display a board, run different AI agents, and present systematic experiments comparing win rates, node expansion, and execution time.

## Setup

Run the cell below to install dependencies and clone the repository (for Google Colab).

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    !pip install numpy matplotlib -q
    if not os.path.exists("CptS440-Connect-Four-Agent"):
        !git clone https://github.com/IsaiahD9402/CptS440-Connect-Four-Agent.git
    os.chdir("CptS440-Connect-Four-Agent")
    print("Running in Colab — repo cloned and ready.")
else:
    print("Running locally.")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from connect_four import ConnectFourBoard, PLAYER_1, PLAYER_2
from ai import (
    get_ai_move,
    get_ab_move,
    get_id_move,
    get_random_move,
    SearchStats,
    clear_transposition_table,
    evaluate,
    simple_evaluate,
    minimax,
    alphabeta,
)
from arena import run_arena, play_game, print_summary

## 1. Board Basics

The board is a 6-row by 7-column NumPy array. Pieces drop to the lowest empty row in a column.

In [ ]:
board = ConnectFourBoard()
print("Empty board:")
print(board.to_string())

board.drop(3, PLAYER_1)
board.drop(3, PLAYER_2)
board.drop(2, PLAYER_1)
board.drop(4, PLAYER_2)
board.drop(2, PLAYER_1)

print("\nAfter some moves:")
print(board.to_string())

## 2. Agent Demonstrations

We compare three agents on the same board position: Minimax (depth 4), Alpha-Beta (depth 6), and Iterative Deepening (2s budget).

In [ ]:
board = ConnectFourBoard()
board.drop(3, PLAYER_1)
board.drop(2, PLAYER_2)
board.drop(3, PLAYER_1)
board.drop(2, PLAYER_2)
print(board.to_string())

for name, fn, kwargs in [
    ("Minimax (depth 4)", get_ai_move, {"depth": 4}),
    ("Alpha-Beta (depth 6)", get_ab_move, {"depth": 6}),
    ("Iterative Deepening (2s)", get_id_move, {"max_depth": 10, "time_limit": 2.0}),
]:
    stats = SearchStats()
    clear_transposition_table()
    t0 = time.perf_counter()
    move = fn(board, PLAYER_1, stats=stats, **kwargs)
    elapsed = time.perf_counter() - t0
    print(f"{name:30s}  move={move}  nodes={stats.nodes:>8,}  time={elapsed:.3f}s")

## 3. Experiment 1 — Win Rate

We pit each agent against the random baseline over 100 games (50 for the slower matchups) and plot win/draw rates.

In [ ]:
matchups = [
    ("AB d6 vs Random", "ab", "random", {"depth": 6}, {}, 100),
    ("Minimax d4 vs Random", "minimax", "random", {"depth": 4}, {}, 100),
    ("AB d6 vs Minimax d4", "ab", "minimax", {"depth": 6}, {"depth": 4}, 50),
    ("ID 2s vs Random", "id", "random", {"max_depth": 12, "time_limit": 2.0}, {}, 50),
]

win_data = {}
for label, p1, p2, kw1, kw2, n in matchups:
    print(f"Running {label} ({n} games)...")
    results = run_arena(p1, p2, n, kw1, kw2, verbose=False)
    p1_wins = sum(1 for r in results if r["winner"] == 1)
    p2_wins = sum(1 for r in results if r["winner"] == 2)
    draws = sum(1 for r in results if r["winner"] == 0)
    win_data[label] = (p1_wins / n * 100, p2_wins / n * 100, draws / n * 100)
    print(f"  P1={p1_wins}/{n}  P2={p2_wins}/{n}  Draw={draws}/{n}\n")

In [ ]:
labels = list(win_data.keys())
p1_pcts = [win_data[l][0] for l in labels]
p2_pcts = [win_data[l][1] for l in labels]
draw_pcts = [win_data[l][2] for l in labels]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, p1_pcts, width, label="P1 Win %")
ax.bar(x, p2_pcts, width, label="P2 Win %")
ax.bar(x + width, draw_pcts, width, label="Draw %")
ax.set_ylabel("Percentage")
ax.set_title("Experiment 1: Win Rate by Matchup")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.legend()
ax.set_ylim(0, 110)
for i, v in enumerate(p1_pcts):
    ax.text(i - width, v + 1, f"{v:.0f}%", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("win_rate_chart.png", dpi=150)
plt.show()

## 4. Experiment 2 — Node Expansion: Minimax vs Alpha-Beta

From the same board position, we compare the number of nodes expanded by plain Minimax vs Alpha-Beta at depths 4, 5, and 6.

In [ ]:
board = ConnectFourBoard()
board.drop(3, PLAYER_1)
board.drop(4, PLAYER_2)
board.drop(3, PLAYER_1)
board.drop(2, PLAYER_2)

depths = [4, 5, 6]
mm_nodes = []
ab_nodes = []

for d in depths:
    stats_mm = SearchStats()
    minimax(board, d, True, PLAYER_1, stats_mm)
    mm_nodes.append(stats_mm.nodes)

    stats_ab = SearchStats()
    clear_transposition_table()
    alphabeta(board, d, float("-inf"), float("inf"), True, PLAYER_1, stats_ab)
    ab_nodes.append(stats_ab.nodes)

    print(f"Depth {d}: Minimax={stats_mm.nodes:>8,}  Alpha-Beta={stats_ab.nodes:>8,}  "
          f"Reduction={100*(1 - stats_ab.nodes/stats_mm.nodes):.1f}%")

In [ ]:
x = np.arange(len(depths))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, mm_nodes, width, label="Minimax")
bars2 = ax.bar(x + width/2, ab_nodes, width, label="Alpha-Beta")
ax.set_xlabel("Search Depth")
ax.set_ylabel("Nodes Expanded")
ax.set_title("Experiment 2: Node Expansion — Minimax vs Alpha-Beta")
ax.set_xticks(x)
ax.set_xticklabels([str(d) for d in depths])
ax.legend()
ax.bar_label(bars1, fmt="{:,.0f}", fontsize=8)
ax.bar_label(bars2, fmt="{:,.0f}", fontsize=8)
plt.tight_layout()
plt.savefig("node_expansion_chart.png", dpi=150)
plt.show()

## 5. Experiment 3 — Execution Time vs Depth

We measure the average time per move for Alpha-Beta at depths 2 through 8, and compare with Iterative Deepening under a fixed time budget.

In [ ]:
board = ConnectFourBoard()
board.drop(3, PLAYER_1)
board.drop(4, PLAYER_2)

test_depths = list(range(2, 9))
ab_times = []

for d in test_depths:
    clear_transposition_table()
    t0 = time.perf_counter()
    get_ab_move(board, PLAYER_1, depth=d)
    elapsed = time.perf_counter() - t0
    ab_times.append(elapsed)
    print(f"AB depth {d}: {elapsed:.4f}s")

print()
for tl in [0.5, 1.0, 2.0, 5.0]:
    clear_transposition_table()
    t0 = time.perf_counter()
    get_id_move(board, PLAYER_1, max_depth=20, time_limit=tl)
    elapsed = time.perf_counter() - t0
    print(f"ID time_limit={tl:.1f}s: actual={elapsed:.4f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(test_depths, ab_times, "o-", linewidth=2, markersize=6, label="Alpha-Beta (fixed depth)")
ax.set_xlabel("Search Depth")
ax.set_ylabel("Time per Move (seconds)")
ax.set_title("Experiment 3: Execution Time vs Search Depth")
ax.set_xticks(test_depths)
ax.legend()
ax.grid(True, alpha=0.3)
for d, t in zip(test_depths, ab_times):
    ax.annotate(f"{t:.3f}s", (d, t), textcoords="offset points", xytext=(0, 10),
               ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("execution_time_chart.png", dpi=150)
plt.show()

## Summary

- **Minimax** provides a correct baseline but explores the full game tree up to the depth limit, making it slow beyond depth 4-5.
- **Alpha-Beta pruning** dramatically reduces the number of nodes explored (typically 60-90% fewer) while producing identical results, enabling deeper search in the same time.
- **Move ordering** (center-first) improves alpha-beta cutoff rates by trying the most promising moves first.
- **Iterative deepening** provides anytime behavior: the agent always has a best move available and uses as much depth as the time budget allows.
- **The transposition table** caches previously evaluated positions to avoid redundant computation across branches.
- **The evaluation function** uses positional weights and sliding window-of-4 scoring to assess board quality without searching deeper.